<a href="https://colab.research.google.com/github/mirikrupkin/foldcheck-rna/blob/main/foldcheck_rna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FoldCheck-RNA

**Compare an RNA 3D structure prediction with an experimental structure from the PDB.** The tool matches nucleotides by sequence alignment, superimposes the prediction on the experimental structure, and reports backbone RMSD, matched nucleotides and sequence identity, with a side-by-side 3D view.

**Author:** [Dr. Miri Krupkin](https://linkedin.com/in/mirikrupkin) (Applied AI Research Scientist & Computational Biologist) &nbsp;|&nbsp; **Version:** 0.0.1 (2026-09-30)

## How to run
1. **Runtime → Run all.** Running needs a free Google account.
2. Scroll to the last cell. It asks for input: a demo number (1-4), or the URL of your own prediction file (raw mmCIF or PDB), then the 4-letter PDB ID of the experimental structure. If your file name starts with the PDB ID (for example `1y26_model.cif`), the notebook suggests it and Enter accepts it. Leave the PDB ID blank (or type `none` when a suggestion is offered) to view your prediction alone, without a comparison. **Press Enter to run the 2OIU demo.** A mistyped choice, an unusable URL or an unknown PDB ID also runs the 2OIU demo, and says so.
3. Results appear below that cell. An HTML report is saved in `assets/report/` (Colab file browser, left panel). Download it before you close the session, because Colab sessions are temporary.

## What's new in 0.0.1
- 2D structure drawings and distance matrices for the prediction and the experimental structure (*experimental*: base pairs are called by a distance heuristic, not by a base-pair annotator).
- Clear error messages when no RNA, or too few nucleotides, can be matched; built-in tests that stop the notebook if they fail.
- Prediction-only view: give a prediction URL and leave the PDB ID blank (or type `none`) to see your model alone (3D view colored by confidence, 2D drawing, distance matrix).
- Earlier versions: side-by-side 3D view and RMSD.

## Status
Early version. Tested on Google Colab on 2026-09-30 (Python 3.13.15): the 8 built-in tests pass, and options 1-5, custom mmCIF and PDB-format URLs, the prediction-only view and the automatic demo run for unusable input ran end to end. A malformed or unknown PDB ID, or a URL that cannot be used, runs the 2OIU demo instead (your prediction is then not used).

**Read before interpreting results:** agreement with one experimental structure is not proof that a prediction is correct. Only the longest RNA chain of each file is used, and NMR entries use model 1 only. See the README for all known limitations.

## Step 1: Setup
Installs dependencies (versions are not pinned yet) and creates working folders. ViennaRNA is used only to draw 2D diagrams.

In [ ]:
print("[INFO] Installing dependencies (versions not pinned)...")
!pip install biopython py3Dmol pandas pytest scipy numpy pyyaml matplotlib seaborn ViennaRNA -q
print("[INFO] Dependency installation finished.")

import os

for folder in ["src", "data", "configs", "tests", "assets"]:
    os.makedirs(folder, exist_ok=True)
print("[INFO] Working folders ready: src, data, configs, tests, assets")


## Step 2: `src/constants.py` (residue names)
Registry that maps residue names (including modified nucleotides) to A/C/G/U, and a list of ion/solvent names to ignore. Known issue: `ADE` and `THY` are also ligand/DNA-like names (planned fix RNA-1.1).

In [ ]:
%%writefile src/constants.py

RNA_HETATM_RES = {
    # Canonical bases and aliases. T/THY (DNA thymine) map to U; ADE is also a free-ligand name (known issue RNA-1.1)
    'A': 'A', 'C': 'C', 'G': 'G', 'U': 'U', 'T': 'U',
    'ADE': 'A', 'CYT': 'C', 'GUA': 'G', 'URI': 'U', 'THY': 'U',

    # Modified nucleotides mapped to their parent base (each mapping unverified)
    'OMC': 'C', 'OMU': 'U', 'OMA': 'A', 'OMG': 'G', '2OM': 'U',
    '5MC': 'C', '5MU': 'U', '2MG': 'G', 'M2G': 'G', '7MG': 'G',
    '1MA': 'A', '2MA': 'A', '1MG': 'G', '2M2': 'G', '6MZ': 'A',

    # Pseudouridine, thio and reduced modifications (mapping unverified)
    'PSU': 'U', 'H2U': 'U', '4SU': 'U', '5BU': 'U', '5IU': 'U',
    'YYG': 'G', 'YG': 'G', 'INP': 'A', 'D': 'U', '3AU': 'U'
}

SOLVENT_IONS = {'HOH', 'MG', 'MN', 'K', 'NA', 'CL', 'ZN', 'CA', 'SO4', 'PO4', 'EDTA', 'ACT'}


## Step 3: `configs/config.yaml`
Only the `alignment` block is read by the code.

In [ ]:
%%writefile configs/config.yaml
project:
  name: "foldcheck-rna"
  version: "0.0.1"

# The 'defaults' block is not read by any code in version 0.0.1.
defaults:
  mode: "2"
  reference_pdb: "1Y26"
  trim_excess: true

alignment:
  open_gap_score: -10.0
  extend_gap_score: -0.5
  match_score: 1.0
  mismatch_score: -2.0


## Step 4: `src/fetcher.py` (download helper)
Downloads a file if it is not already present. Errors (bad URL, HTTP error) are raised, not hidden.

In [ ]:
%%writefile src/fetcher.py
import os
import urllib.request


def download_file(url: str, output_path: str):
    """Download url to output_path unless the file already exists. Raises on HTTP or network errors."""
    if not os.path.exists(output_path):
        urllib.request.urlretrieve(url, output_path)
    return output_path


## Step 5: `src/alignment.py` (matching and superposition)
Pairs nucleotides by global sequence alignment and superimposes the prediction on the experimental structure (least squares, Biopython `Superimposer`). Raises an error if fewer than 3 pairs match, and warns below 10. Known limitations: longest chain only (RNA-1.3); P and C4' atoms can be mixed (RNA-1.4).

In [ ]:
%%writefile src/alignment.py
import os
import warnings
import yaml
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from Bio.PDB import Superimposer, PDBIO
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.PDBParser import PDBParser
from Bio.Align import PairwiseAligner
from src.constants import RNA_HETATM_RES, SOLVENT_IONS

# Matched nucleotide pairs: raise below MIN_PAIRS_ERROR, warn below MIN_PAIRS_WARN
MIN_PAIRS_ERROR = 3
MIN_PAIRS_WARN = 10


def load_alignment_config():
    config_path = "configs/config.yaml"
    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            cfg = yaml.safe_load(f)
            return cfg.get("alignment", {})
    return {"open_gap_score": -10.0, "extend_gap_score": -0.5, "match_score": 1.0, "mismatch_score": -2.0}

def parse_structure_file(filepath: str, structure_id: str = "model", keep_longest_chain_only: bool = True):
    ext = os.path.splitext(filepath)[1].lower()
    parser = MMCIFParser(QUIET=True) if ext == ".cif" else PDBParser(QUIET=True)
    structure = parser.get_structure(structure_id, filepath)

    model = structure[0] if len(structure) > 0 else structure

    for idx, chain in enumerate(model):
        if chain.id in [' ', '', '\n'] or chain.id is None:
            chain.id = chr(65 + (idx % 26))

    for chain in model:
        res_to_detach = [res.id for res in chain if res.get_resname().strip() in SOLVENT_IONS]
        for r_id in res_to_detach:
            chain.detach_child(r_id)

    # Known limitation (RNA-1.3): all chains except the longest are dropped without a message.
    if keep_longest_chain_only and len(list(model)) > 1:
        chains = list(model)
        valid_chain_lengths = []
        for c in chains:
            count = sum(1 for res in c if res.get_resname().strip() in RNA_HETATM_RES)
            valid_chain_lengths.append((count, c.id))

        valid_chain_lengths.sort(reverse=True, key=lambda x: x[0])
        longest_chain_id = valid_chain_lengths[0][1]

        chains_to_detach = [c.id for c in chains if c.id != longest_chain_id]
        for c_id in chains_to_detach:
            model.detach_child(c_id)

    return structure

def count_valid_nucleotides(structure):
    model = structure[0] if len(structure) > 0 else structure
    count = 0
    for chain in model:
        for res in chain:
            resname = res.get_resname().strip()
            if resname in RNA_HETATM_RES and resname not in SOLVENT_IONS:
                count += 1
    return count

def get_biophysical_backbone_atom(residue):
    """Return the P atom if present, else C4' (or C4*), else None.
    The choice is made per residue and independently for each structure, so a P atom can be paired with a C4' atom
    (known issue RNA-1.4)."""
    if residue.has_id('P'):
        return residue['P']
    elif residue.has_id("C4'"):
        return residue["C4'"]
    elif residue.has_id("C4*"):
        return residue["C4*"]
    return None

def calculate_structural_rmsd_and_deviation(ref_pdb_path: str, target_pdb_path: str, trim_excess: bool = True, keep_longest_chain_only: bool = True):
    """Match nucleotides by global sequence alignment, superimpose the prediction on the reference and return
    (rmsd, matched_pairs, sequence_identity_percent_of_matched, aligned_prediction_file, aligned_reference_file,
    deviation_table, notes). Raises ValueError if fewer than MIN_PAIRS_ERROR pairs can be matched.
    trim_excess is accepted for call compatibility but is not used."""
    base_name = os.path.basename(ref_pdb_path).replace("EXP_", "").split(".")[0].upper()
    if len(base_name) != 4:
        base_name = "RNA"

    ref_structure = parse_structure_file(ref_pdb_path, "experimental", keep_longest_chain_only=keep_longest_chain_only)
    target_structure = parse_structure_file(target_pdb_path, "predicted", keep_longest_chain_only=keep_longest_chain_only)

    ref_model = ref_structure[0] if len(ref_structure) > 0 else ref_structure
    target_model = target_structure[0] if len(target_structure) > 0 else target_structure

    def get_rna_residues(chain):
        return [res for res in chain if res.get_resname().strip() in RNA_HETATM_RES and res.get_resname().strip() not in SOLVENT_IONS]

    ref_chains = [c for c in ref_model if len(get_rna_residues(c)) > 0]
    target_chains = [c for c in target_model if len(get_rna_residues(c)) > 0]
    if not ref_chains:
        raise ValueError(f"No RNA nucleotides were found in the experimental structure ({ref_pdb_path}). "
                         "Check the PDB ID and that the entry contains RNA.")
    if not target_chains:
        raise ValueError(f"No RNA nucleotides were found in the prediction file ({target_pdb_path}). "
                         "Check that the file is an RNA structure and that the URL points to the coordinate file itself.")

    ref_atoms_matched = []
    target_atoms_matched = []

    cfg = load_alignment_config()
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    aligner.open_gap_score = cfg.get("open_gap_score", -10.0)
    aligner.extend_gap_score = cfg.get("extend_gap_score", -0.5)
    aligner.match_score = cfg.get("match_score", 1.0)
    aligner.mismatch_score = cfg.get("mismatch_score", -2.0)

    score_matrix = np.zeros((len(ref_chains), len(target_chains)))

    for i, r_chain in enumerate(ref_chains):
        for j, t_chain in enumerate(target_chains):
            r_seq = "".join([RNA_HETATM_RES.get(res.get_resname().strip(), 'N') for res in get_rna_residues(r_chain)])
            t_seq = "".join([RNA_HETATM_RES.get(res.get_resname().strip(), 'N') for res in get_rna_residues(t_chain)])

            if not r_seq or not t_seq:
                continue

            alignment = aligner.align(r_seq, t_seq)
            if alignment:
                score_matrix[i, j] = alignment[0].score

    row_ind, col_ind = linear_sum_assignment(score_matrix, maximize=True)

    for r_idx, t_idx in zip(row_ind, col_ind):
        r_chain = ref_chains[r_idx]
        t_chain = target_chains[t_idx]

        r_res = get_rna_residues(r_chain)
        t_res = get_rna_residues(t_chain)

        r_seq = "".join([RNA_HETATM_RES.get(res.get_resname().strip(), 'N') for res in r_res])
        t_seq = "".join([RNA_HETATM_RES.get(res.get_resname().strip(), 'N') for res in t_res])

        alignment = aligner.align(r_seq, t_seq)[0]
        r_indices = alignment.aligned[0]
        t_indices = alignment.aligned[1]

        for (r_start, r_end), (t_start, t_end) in zip(r_indices, t_indices):
            for i in range(r_end - r_start):
                r_r = r_res[r_start + i]
                t_r = t_res[t_start + i]
                r_atom = get_biophysical_backbone_atom(r_r)
                t_atom = get_biophysical_backbone_atom(t_r)
                if r_atom is not None and t_atom is not None:
                    ref_atoms_matched.append(r_atom)
                    target_atoms_matched.append(t_atom)

    min_len = len(ref_atoms_matched)
    if min_len < MIN_PAIRS_ERROR:
        raise ValueError(
            f"Only {min_len} nucleotide pair(s) could be matched between the prediction and the experimental structure "
            f"(minimum {MIN_PAIRS_ERROR}). Check that the PDB ID is the right entry and that both files contain the same RNA."
        )
    if min_len < MIN_PAIRS_WARN:
        warnings.warn(f"Only {min_len} nucleotide pairs were matched; the RMSD is based on very few atoms.")

    identical_count = sum(
        1 for r, t in zip(ref_atoms_matched, target_atoms_matched)
        if RNA_HETATM_RES.get(r.get_parent().get_resname().strip(), 'N') ==
           RNA_HETATM_RES.get(t.get_parent().get_resname().strip(), 'N')
    )

    superimposer = Superimposer()
    superimposer.set_atoms(ref_atoms_matched, target_atoms_matched)
    superimposer.apply(target_structure.get_atoms())
    rms_val = superimposer.rms

    os.makedirs("data", exist_ok=True)
    aligned_ref_filename = os.path.join("data", f"{base_name}_reference_aligned.pdb")
    io_ref = PDBIO()
    io_ref.set_structure(ref_structure)
    io_ref.save(aligned_ref_filename)

    aligned_filename = os.path.join("data", f"{base_name}_prediction_aligned.pdb")
    io = PDBIO()
    io.set_structure(target_structure)
    io.save(aligned_filename)

    deviation_records = []
    for r_atom, t_atom in zip(ref_atoms_matched, target_atoms_matched):
        diff_vector = r_atom.get_coord() - t_atom.get_coord()
        distance = float((diff_vector ** 2).sum() ** 0.5)
        res = t_atom.get_parent()
        deviation_records.append({
            "Nucleotide_ID": res.get_id()[1],
            "Nucleotide": res.get_resname().strip(),
            "Backbone_Deviation_Angstroms": round(distance, 3)
        })

    seq_id = (identical_count / min_len) * 100 if min_len > 0 else 0
    trimmed_info = ("Superposition atoms: P if present, else C4' (chosen per residue on each side). "
                    "Only the longest RNA chain of each file is used; nothing else is trimmed.")

    df_dev = pd.DataFrame(deviation_records)
    return rms_val, min_len, seq_id, aligned_filename, aligned_ref_filename, df_dev, trimmed_info


## Step 6: `src/metrics.py` (confidence values)
Reads the B-factor column of each nucleotide's backbone atom. This is pLDDT only if the predictor stored it there. Bands are plain numeric ranges. The mean is `None` when no value can be read.

In [ ]:
%%writefile src/metrics.py
import warnings
import numpy as np
import pandas as pd
from src.alignment import parse_structure_file, get_biophysical_backbone_atom
from src.constants import RNA_HETATM_RES, SOLVENT_IONS


def plddt_band(value):
    """Numeric band label for a per-nucleotide confidence value (no RNA-specific interpretation is claimed)."""
    if value is None or np.isnan(value):
        return "Not available (no backbone atom found)"
    if value >= 90:
        return "pLDDT >= 90"
    if value >= 70:
        return "70 <= pLDDT < 90"
    if value >= 50:
        return "50 <= pLDDT < 70"
    return "pLDDT < 50"


def analyze_alphafold_model(pdb_path: str):
    """Read per-nucleotide confidence from the B-factor column of the backbone atom.

    Returns (table, mean_confidence); mean_confidence is None if no value could be read.
    The B-factor column holds pLDDT only for predictors that store it there. For other files
    (for example experimental structures) the numbers are not pLDDT.
    """
    structure = parse_structure_file(pdb_path, "af_rna_model")
    records = []
    values = []
    for model in structure:
        for chain in model:
            for residue in chain:
                resname = residue.get_resname().strip()
                if resname in RNA_HETATM_RES and resname not in SOLVENT_IONS:
                    atom = get_biophysical_backbone_atom(residue)
                    value = atom.get_bfactor() if atom is not None else float("nan")
                    if not np.isnan(value):
                        values.append(value)
                    records.append({
                        "Nucleotide_ID": residue.get_id()[1],
                        "Nucleotide": resname,
                        "pLDDT": value,
                        "Confidence_Band": plddt_band(value),
                    })
    if values and max(values) > 100.0:
        warnings.warn(
            f"B-factor values above 100 found (max {max(values):.1f}). This file may be an experimental structure, "
            "not a prediction with pLDDT in the B-factor column."
        )
    table = pd.DataFrame(records)
    mean_value = round(float(np.mean(values)), 2) if values else None
    return table, mean_value


## Step 7: Tests
Small tests on synthetic coordinates (no network): identity, rotation/translation invariance, and clear errors when nothing (or too little) matches.

In [ ]:
%%writefile tests/test_pipeline.py
import numpy as np
import pytest
from scipy.spatial.transform import Rotation

from src.constants import RNA_HETATM_RES
from src.alignment import calculate_structural_rmsd_and_deviation
from src.metrics import analyze_alphafold_model


def write_pdb(path, resnames, coords, atom="P", bfactor=50.0):
    """Write one chain 'A' with one atom per residue in fixed-column PDB format."""
    lines = []
    for i, (resname, (x, y, z)) in enumerate(zip(resnames, coords), start=1):
        name = f" {atom:<3s}"
        lines.append(
            f"ATOM  {i:5d} {name} {resname:>3s} A{i:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.0:6.2f}{bfactor:6.2f}          {atom[0]:>2s}"
        )
    lines.append("END")
    path.write_text("\n".join(lines) + "\n")


def random_walk(n, seed=0):
    rng = np.random.default_rng(seed)
    steps = rng.normal(size=(n, 3))
    steps /= np.linalg.norm(steps, axis=1)[:, None]
    return np.cumsum(steps * 5.9, axis=0)


def test_nucleotide_dictionary():
    assert RNA_HETATM_RES.get("A") == "A"
    assert RNA_HETATM_RES.get("U") == "U"
    assert RNA_HETATM_RES.get("OMC") == "C"
    assert RNA_HETATM_RES.get("UNKNOWN") is None


def test_identical_structures(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    coords = random_walk(20)
    seq = list("ACGU" * 5)
    write_pdb(tmp_path / "EXP_TEST.pdb", seq, coords)
    write_pdb(tmp_path / "pred.pdb", seq, coords)
    rmsd, n_pairs, identity, _, _, deviations, _ = calculate_structural_rmsd_and_deviation(
        str(tmp_path / "EXP_TEST.pdb"), str(tmp_path / "pred.pdb"))
    assert rmsd < 0.01
    assert n_pairs == 20
    assert identity == pytest.approx(100.0)
    assert len(deviations) == 20
    assert deviations["Backbone_Deviation_Angstroms"].max() < 0.01


def test_rotation_and_translation_invariance(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    coords = random_walk(20, seed=1)
    moved = Rotation.from_rotvec([0.4, -1.1, 0.7]).apply(coords) + np.array([10.0, -5.0, 3.0])
    seq = list("GGCUAAUCGAUCGGAUCCGA")
    write_pdb(tmp_path / "EXP_TEST.pdb", seq, coords)
    write_pdb(tmp_path / "pred.pdb", seq, moved)
    rmsd, n_pairs, _, _, _, _, _ = calculate_structural_rmsd_and_deviation(
        str(tmp_path / "EXP_TEST.pdb"), str(tmp_path / "pred.pdb"))
    assert rmsd < 0.01
    assert n_pairs == 20


def test_protein_prediction_raises_clear_error(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    coords = random_walk(20)
    write_pdb(tmp_path / "EXP_TEST.pdb", list("ACGU" * 5), coords)
    write_pdb(tmp_path / "pred.pdb", ["ALA"] * 20, coords, atom="CA")
    with pytest.raises(ValueError, match="No RNA nucleotides"):
        calculate_structural_rmsd_and_deviation(str(tmp_path / "EXP_TEST.pdb"), str(tmp_path / "pred.pdb"))


def test_too_few_pairs_raises(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    coords = random_walk(2)
    write_pdb(tmp_path / "EXP_TEST.pdb", list("AC"), coords)
    write_pdb(tmp_path / "pred.pdb", list("AC"), coords)
    with pytest.raises(ValueError, match="Only 2"):
        calculate_structural_rmsd_and_deviation(str(tmp_path / "EXP_TEST.pdb"), str(tmp_path / "pred.pdb"))


def test_few_pairs_warns(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    coords = random_walk(5)
    write_pdb(tmp_path / "EXP_TEST.pdb", list("ACGUA"), coords)
    write_pdb(tmp_path / "pred.pdb", list("ACGUA"), coords)
    with pytest.warns(UserWarning, match="Only 5"):
        calculate_structural_rmsd_and_deviation(str(tmp_path / "EXP_TEST.pdb"), str(tmp_path / "pred.pdb"))


def test_mean_confidence_is_none_when_no_backbone_atom(tmp_path):
    coords = random_walk(12)
    write_pdb(tmp_path / "pred.pdb", list("ACGU" * 3), coords, atom="N1")
    table, mean_value = analyze_alphafold_model(str(tmp_path / "pred.pdb"))
    assert mean_value is None
    assert len(table) == 12


def test_mean_confidence_value(tmp_path):
    coords = random_walk(12)
    write_pdb(tmp_path / "pred.pdb", list("ACGU" * 3), coords, bfactor=77.5)
    _, mean_value = analyze_alphafold_model(str(tmp_path / "pred.pdb"))
    assert mean_value == pytest.approx(77.5)


## Step 8: Run the tests
Stops the notebook with an error if any test fails.

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-v"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Tests failed - see the output above."


## Step 9: Workflow figure
Draws a schematic of what the tool does in this version.

In [ ]:
import os
import matplotlib.pyplot as plt

def generate_workflow_figure():
    fig, ax = plt.subplots(figsize=(13, 9), dpi=300)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.axis('off')

    box_props = dict(boxstyle="round,pad=0.8", facecolor="#f8f9fa", edgecolor="#343a40", linewidth=1.5)
    core_box_props = dict(boxstyle="round,pad=1.0", facecolor="#e9ecef", edgecolor="#212529", linewidth=2)
    arrow_props = dict(arrowstyle="-|>", lw=2, color="#495057", mutation_scale=15)
    large_arrow_props = dict(arrowstyle="-|>", lw=2.5, color="#495057", mutation_scale=22)

    # Nodes
    ax.text(50, 91, "Inputs\n(prediction file + experimental PDB ID)", ha='center', va='center', fontsize=9, bbox=box_props, wrap=True)
    ax.text(25, 70, "Prediction file\n(direct URL to mmCIF / PDB)", ha='center', va='center', fontsize=8.5, bbox=box_props, wrap=True)
    ax.text(25, 50, "Predicted structure\n(any predictor; B-factors read as pLDDT)", ha='center', va='center', fontsize=8.5, bbox=box_props, wrap=True)
    ax.text(75, 60, "Experimental structure\n(downloaded from RCSB by PDB ID)", ha='center', va='center', fontsize=8.5, bbox=box_props, wrap=True)

    core_text = (
        "FoldCheck-RNA processing steps\n\n"
        "• Longest-chain selection (current limitation)\n"
        "• Modified nucleotides mapped to parent base\n"
        "• Global sequence alignment for residue matching\n"
        "• Least-squares superposition on P (else C4\') atoms"
    )
    ax.text(50, 27, core_text, ha='center', va='center', fontsize=9, bbox=core_box_props, wrap=True)

    output_text = (
        "Reporting & visualizations\n\n"
        "• Side-by-side 3D viewer (py3Dmol)\n"
        "• 2D drawings from a distance heuristic (experimental)\n"
        "• Distance matrices (experimental)\n"
        "• JSON summary and HTML snapshot"
    )
    ax.text(50, 6, output_text, ha='center', va='center', fontsize=8.5, bbox=box_props, wrap=True)

    # Connectors
    ax.annotate("", xy=(25, 79), xytext=(40, 87), arrowprops=arrow_props)
    ax.annotate("", xy=(75, 68), xytext=(60, 87), arrowprops=arrow_props)
    ax.annotate("", xy=(25, 57), xytext=(25, 63), arrowprops=large_arrow_props)
    ax.annotate("", xy=(38, 41), xytext=(31, 46), arrowprops=arrow_props)
    ax.annotate("", xy=(62, 41), xytext=(68, 53), arrowprops=arrow_props)
    ax.annotate("", xy=(50, 13), xytext=(50, 17), arrowprops=large_arrow_props)

    ax.axhline(y=97, xmin=0.1, xmax=0.9, color="#dee2e6", linewidth=1)
    ax.text(50, 98, "FoldCheck-RNA Multi-Modal Validation Architecture", ha='center', va='bottom', fontsize=12, fontweight='bold', color="#212529")

    os.makedirs("assets/figures", exist_ok=True, mode=0o755)
    output_path = "assets/figures/figure_1_workflow_schematic.png"
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"[INFO] Workflow figure saved to: {output_path}")

generate_workflow_figure()


## Step 10: `src/contact_map.py` (distance matrices, experimental)
Distance matrix between one backbone atom per nucleotide. Experimental: the atom type can differ between residues (RNA-1.4).

In [ ]:
%%writefile src/contact_map.py
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.PDBParser import PDBParser
from src.constants import RNA_HETATM_RES, SOLVENT_IONS

def compute_and_render_contact_map(filepath: str, detected_pdb: str):
    """Experimental. Distance matrix between one backbone atom per nucleotide (P, else C4'; the atom type is chosen
    per residue, known issue RNA-1.4). Saves a heatmap figure and a CSV of pairs closer than 12 A."""
    ext = os.path.splitext(filepath)[1].lower()
    parser = MMCIFParser(QUIET=True) if ext == ".cif" else PDBParser(QUIET=True)
    structure = parser.get_structure("rna_topology", filepath)
    model = structure[0] if len(structure) > 0 else structure

    residues_data = []
    for chain in model:
        for res in chain:
            resname = res.get_resname().strip()
            if resname in RNA_HETATM_RES and resname not in SOLVENT_IONS:
                atom = res['P'] if res.has_id('P') else (res["C4'"] if res.has_id("C4'") else None)
                if atom is not None:
                    residues_data.append({
                        "Chain": chain.id,
                        "Residue_ID": res.get_id()[1],
                        "Residue_Name": RNA_HETATM_RES.get(resname, 'N'),
                        "Coord": atom.get_coord()
                    })

    n = len(residues_data)
    if n == 0:
        return None, None

    coords = np.array([r["Coord"] for r in residues_data])
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff**2, axis=-1))

    # Save Heatmap Figure
    os.makedirs("assets/figures", exist_ok=True)
    os.makedirs("assets/tables", exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
    sns.heatmap(
        dist_matrix,
        cmap="mako_r",
        cbar_kws={'label': 'Inter-Nucleotide Distance (Å)'},
        ax=ax,
        vmin=0,
        vmax=30
    )
    ax.set_title(f"Inter-nucleotide distance matrix (experimental)\nStructure: {detected_pdb}", fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel("Nucleotide Index", fontsize=10, fontweight='semibold')
    ax.set_ylabel("Nucleotide Index", fontsize=10, fontweight='semibold')

    figure_path = f"assets/figures/figure_2_contact_map_{detected_pdb}.png"
    plt.savefig(figure_path, bbox_inches='tight', dpi=300)
    plt.close(fig)

    # Serialize Contact Records Table
    contact_records = []
    contact_threshold = 12.0
    for i in range(n):
        for j in range(i + 3, n): # Exclude local neighbors
            if dist_matrix[i, j] < contact_threshold:
                contact_records.append({
                    "Target": detected_pdb,
                    "Residue_1_ID": residues_data[i]["Residue_ID"],
                    "Residue_1_Name": residues_data[i]["Residue_Name"],
                    "Residue_2_ID": residues_data[j]["Residue_ID"],
                    "Residue_2_Name": residues_data[j]["Residue_Name"],
                    "Distance_Angstroms": round(float(dist_matrix[i, j]), 2)
                })

    df_contacts = pd.DataFrame(contact_records)
    csv_output_path = f"assets/tables/contact_interactions_{detected_pdb}.csv"
    df_contacts.to_csv(csv_output_path, index=False)

    return figure_path, csv_output_path


## Step 11: `src/topology_2d.py` (2D drawings, experimental)
Pairs are called by a distance heuristic, not by a base-pair annotator; results can be wrong (RNA-3.1, RNA-3.3). ViennaRNA only draws.

In [ ]:
%%writefile src/topology_2d.py
import os
import base64
import numpy as np
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.PDBParser import PDBParser
from src.constants import RNA_HETATM_RES, SOLVENT_IONS


try:
    import RNA
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "ViennaRNA", "-q"])
    import RNA

def extract_sequence_and_base_atoms(filepath: str):
    """Extract the sequence and the non-backbone heavy-atom coordinates of each nucleotide-like residue.
    Residue names missing from the registry are assigned a base by name substring (known issue RNA-1.2)."""
    ext = os.path.splitext(filepath)[1].lower()
    parser = MMCIFParser(QUIET=True) if ext == ".cif" else PDBParser(QUIET=True)
    structure = parser.get_structure("coord_extract", filepath)
    model = structure[0] if len(structure) > 0 else structure

    sequence_list = []
    residue_base_atoms = []
    backbone_atoms = {'P', 'OP1', 'OP2', 'O5\'', 'C5\'', 'C4\'', 'C3\'', 'O3\'', 'C2\'', 'C1\'', 'O2\'', 'O4\''}

    for chain in model:
        for res in chain:
            resname = res.get_resname().strip()
            if resname in SOLVENT_IONS:
                continue

            # Registry lookup first; names not in the registry are guessed below (known issue RNA-1.2)
            canonical = RNA_HETATM_RES.get(resname, None)
            if canonical is None:
                # Guess the base from the residue name by substring
                if 'C' in resname: canonical = 'C'
                elif 'U' in resname: canonical = 'U'
                elif 'G' in resname: canonical = 'G'
                elif 'A' in resname: canonical = 'A'
                else:
                    continue # Skip names that contain none of C, U, G, A

            sequence_list.append(canonical)

            base_coords = []
            for atom in res:
                if atom.get_name() not in backbone_atoms and not atom.get_name().startswith('H'):
                    base_coords.append(atom.get_coord())

            residue_base_atoms.append({
                "id": res.get_id()[1],
                "resname": canonical,
                "coords": base_coords
            })

    return "".join(sequence_list), residue_base_atoms

def compute_coordinate_derived_structure(residues):
    """Experimental distance heuristic, NOT a base-pair annotation. Any two residues at least 4 positions apart whose
    closest heavy atoms are within 3.8 A are candidate pairs; pairs are taken greedily by distance. There is no
    complementarity or geometry check, stacking can be called as a pair, and crossing pairs are written as ordinary
    brackets (known issues RNA-3.1, RNA-3.3)."""
    n = len(residues)
    dot_bracket = ['.'] * n
    pairs = []

    for i in range(n):
        for j in range(i + 4, n):  # Minimum hairpin loop span of 4 nucleotides
            res_i = residues[i]["coords"]
            res_j = residues[j]["coords"]

            if not res_i or not res_j:
                continue

            min_dist = float('inf')
            for coord_a in res_i:
                for coord_b in res_j:
                    d = np.linalg.norm(coord_a - coord_b)
                    if d < min_dist:
                        min_dist = d

            # 3.8 A minimum heavy-atom distance (heuristic; no base-identity check)
            if min_dist <= 3.8:
                pairs.append((i, j, min_dist))

    pairs.sort(key=lambda x: x[2])
    paired_indices = set()
    valid_pairs = []

    for i, j, d in pairs:
        if i not in paired_indices and j not in paired_indices:
            valid_pairs.append((i, j))
            paired_indices.add(i)
            paired_indices.add(j)

    for i, j in valid_pairs:
        dot_bracket[i] = '('
        dot_bracket[j] = ')'

    return "".join(dot_bracket), len(valid_pairs)

def generate_2d_topology_svg(pdb_filepath: str, output_svg_path: str):
    """Experimental. Draws an SVG (ViennaRNA) of the pairs called by the distance heuristic."""
    if not os.path.exists(pdb_filepath):
        return None, "", 0, "File not found"

    seq, res_data = extract_sequence_and_base_atoms(pdb_filepath)
    if not seq:
        return None, "", 0, "No valid RNA sequence extracted"

    dot_bracket, pair_count = compute_coordinate_derived_structure(res_data)

    os.makedirs(os.path.dirname(output_svg_path), exist_ok=True)
    RNA.svg_rna_plot(seq, dot_bracket, output_svg_path)

    with open(output_svg_path, "rb") as svg_file:
        svg_base64 = base64.b64encode(svg_file.read()).decode('utf-8')

    return svg_base64, dot_bracket, pair_count, seq


## Step 12: Interactive sandbox
Asks for input (`input()` prompts; form fields are planned). Choose a demo (1-4), or paste the URL of a raw mmCIF/PDB prediction file and then a 4-letter PDB ID (suggested from the file name when it starts with a PDB ID; blank or `none` = view the prediction alone). Known limitations: only the longest chain is used; an unrecognized PDB ID or an unusable URL runs the 2OIU demo instead (your prediction is then not used); demo predictions come from a personal GitHub repository.

In [ ]:
import os
import sys
import json
import base64
import html as html_lib
import re
import urllib.error
import urllib.request
import numpy as np
from IPython.display import display, HTML

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.fetcher import download_file
from src.alignment import calculate_structural_rmsd_and_deviation, parse_structure_file, count_valid_nucleotides
from src.metrics import analyze_alphafold_model
from src.constants import RNA_HETATM_RES, SOLVENT_IONS
from src.topology_2d import generate_2d_topology_svg, extract_sequence_and_base_atoms
from src.contact_map import compute_and_render_contact_map
from Bio import pairwise2
from Bio.PDB import PDBIO

def sanitize_and_validate_custom_url(url: str, output_path: str):
    """Convert a GitHub blob link to a raw link, download it and check that it looks like mmCIF or PDB.
    The file is saved with the extension that matches its content (.cif or .pdb), because the parser is chosen by
    extension. Returns the path that was written."""
    if "github.com" in url and "/blob/" in url:
        url = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    if os.path.exists(output_path):
        os.remove(output_path)
    urllib.request.urlretrieve(url, output_path)

    file_format = None
    with open(output_path, 'r', encoding='utf-8', errors='ignore') as f:
        for _ in range(200):
            line = f.readline()
            if not line:
                break
            stripped = line.strip()
            if stripped.startswith("data_"):
                file_format = "cif"
                break
            if stripped.startswith("ATOM") or stripped.startswith("HETATM"):
                file_format = "pdb"
                break

    if file_format is None:
        raise ValueError(
            f"Invalid structure file downloaded from URL!\n"
            f"The file does not contain recognizable mmCIF ('data_') or PDB ('ATOM') headers.\n"
            f"Please verify that your custom URL points directly to a raw text coordinate file."
        )

    final_path = os.path.splitext(output_path)[0] + "." + file_format
    if final_path != output_path:
        os.replace(output_path, final_path)
    print(f"[INFO] Successfully downloaded and verified structural coordinate file: {final_path}")
    return final_path

def image_to_base64(img_path: str):
    """Encodes an image file into base64 string for inline HTML rendering."""
    if os.path.exists(img_path):
        with open(img_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    return ""

def generate_highlighted_sequence_html(ref_seq: str, pred_seq: str):
    """Globally align two sequences (pairwise2) and return HTML with mismatches highlighted."""
    try:
        alignments = pairwise2.align.globalms(ref_seq, pred_seq, 2.0, -1.0, -2.0, -0.5, one_alignment_only=True)
        if not alignments:
            raise ValueError("the sequence alignment returned no result")
        ref_aln, pred_aln, score, begin, end = alignments[0]
    except Exception as e:
        return ('<div style="font-family: monospace; font-size: 10px; background: #12121c; color: #fca5a5; padding: 10px; '
                'border-radius: 6px; margin-top: 10px; border: 1px solid #333;">'
                f'Sequence comparison not available: {e}</div>')

    ref_html = []
    pred_html = []
    matches = 0
    total_len = max(len(ref_aln), len(pred_aln))

    for r_char, p_char in zip(ref_aln, pred_aln):
        if r_char == '-' or p_char == '-':
            ref_html.append(f"<span style='background: #475569; color: #ffffff; padding: 1px 2px; border-radius: 2px;'>{r_char}</span>")
            pred_html.append(f"<span style='background: #475569; color: #ffffff; padding: 1px 2px; border-radius: 2px;'>{p_char}</span>")
        elif r_char == p_char:
            matches += 1
            ref_html.append(f"<span style='color: #cbd5e1;'>{r_char}</span>")
            pred_html.append(f"<span style='color: #cbd5e1;'>{p_char}</span>")
        else:
            ref_html.append(f"<span style='background: #7f1d1d; color: #fca5a5; padding: 1px 2px; border-radius: 2px;'>{r_char}</span>")
            pred_html.append(f"<span style='background: #7f1d1d; color: #fca5a5; padding: 1px 2px; border-radius: 2px;'>{p_char}</span>")

    identity_pct = (matches / total_len) * 100 if total_len > 0 else 0.0

    box_content = f"""
    <div style="font-family: monospace; font-size: 10px; background: #12121c; padding: 10px; border-radius: 6px; overflow-x: auto; text-align: left; margin-top: 10px; border: 1px solid #333;">
        <div style="color: #94a3b8; margin-bottom: 4px; word-break: break-all;"><strong style="color: #00e5ff;">Ref:</strong>  {"".join(ref_html)}</div>
        <div style="color: #94a3b8; word-break: break-all;"><strong style="color: #00e5ff;">Pred:</strong> {"".join(pred_html)}</div>
        <div style="color: #38bdf8; margin-top: 8px; font-size: 10px;">Identity over alignment columns: <strong>{identity_pct:.1f}%</strong></div>
    </div>
    """
    return box_content

# pLDDT bands for the 3D view: (color, label in the legend, 3Dmol range selector on the B-factor column).
# Listed from low to high so that later styles win on the boundaries (50 -> yellow, 70 -> light blue, 90 -> blue).
PLDDT_BANDS = [
    ("#FF7D45", "&lt;50", "0-50"),
    ("#FFDB13", "50-70", "50-70"),
    ("#65CBF3", "70-90", "70-90"),
    ("#0053D6", "&gt;90", "90-100"),
]
_legend_items = " ".join(f'<span style="color: {c};">&#9632;</span> {label}' for c, label, _ in PLDDT_BANDS)
PLDDT_LEGEND = (
    '<small style="color: #cbd5e1;">Colored by pLDDT (B-factor column):<br>low ' + _legend_items + ' high</small>'
)
COLOR_NOTE_HTML = (
    '<div style="width: 920px; font-family: sans-serif; font-size: 12px; line-height: 1.5; color: #94a3b8; margin-top: 10px;">'
    '<strong style="color: #cbd5e1;">Note on colors:</strong> the colors show the B-factor column of the prediction file. '
    'These are pLDDT values only if the predictor stored them there. Gray = outside 0-100.</div>'
)


def style_by_plddt(view, grid_position=None):
    """Color the cartoon in the four bands of PLDDT_LEGEND. Atoms with a B-factor outside 0-100 stay gray."""
    kw = {} if grid_position is None else {"viewer": grid_position}
    view.setStyle({}, {'cartoon': {'color': '#9aa5b1'}}, **kw)
    for color, _label, selector in PLDDT_BANDS:
        view.setStyle({'b': selector}, {'cartoon': {'color': color}}, **kw)


def first_chain_id(path):
    """Chain ID of the first atom line in a PDB file ('?' if there is none)."""
    with open(path) as handle:
        for line in handle:
            if line.startswith(("ATOM", "HETATM")):
                return line[21].strip() or "?"
    return "?"


DEMO_PDB = "2OIU"
DEMO_URL = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/2oiu_prediction.cif"


def reference_exists(pdb_code: str):
    """Download the RCSB mmCIF file for pdb_code into data/. Returns False if RCSB reports no such entry (HTTP 404)."""
    os.makedirs("data", exist_ok=True)
    try:
        download_file(f"https://files.rcsb.org/download/{pdb_code}.cif", os.path.join("data", f"EXP_{pdb_code}.cif"))
        return True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return False
        raise


def prepare_prediction(url: str):
    """Download and check the user's prediction URL.
    Returns (saved_path, None) if it is a readable structure file that contains RNA, otherwise (None, reason)."""
    os.makedirs("data", exist_ok=True)
    try:
        path = sanitize_and_validate_custom_url(url, os.path.join("data", "target_input.cif"))
        structure = parse_structure_file(path, "pred_input", keep_longest_chain_only=True)
        if count_valid_nucleotides(structure) == 0:
            return None, "the file contains no RNA nucleotides"
        return path, None
    except urllib.error.HTTPError as e:
        return None, f"the server answered HTTP {e.code}"
    except urllib.error.URLError as e:
        return None, f"the address could not be reached: {e.reason}"
    except ValueError:
        return None, "the address did not return an mmCIF or PDB coordinate file"
    except Exception as e:
        return None, f"the file could not be read ({type(e).__name__})"


print("\n" * 2)
print("==================================================")
print("FoldCheck-RNA sandbox (interactive)")
print("==================================================")
print("Choose a demo, paste a URL to your own prediction file, or press Enter:")
print("  [1] Adenine riboswitch aptamer   [PDB: 1Y26]")
print("  [2] Yeast tRNA-Phe               [PDB: 1EHZ]")
print("  [3] THF riboswitch               [PDB: 4LVV]")
print("  [4] Hammerhead ribozyme          [PDB: 2OEU]")
print("  [5] Your own prediction: URL, then PDB ID (blank PDB ID shows the prediction alone; blank URL runs the 2OIU demo)")
print("  (Demo prediction files are loaded from a personal GitHub repository.)")
print("  [Enter] Run the 2OIU demo (a mistyped choice, unusable URL or unknown PDB ID also runs it)")
print("-"*50)

try:
    user_input = input("\nEnter choice [1-5], paste a direct prediction URL, or press Enter for the 2OIU demo: ").strip()
except Exception:
    user_input = None

if user_input is None:
    print("\n[INFO] No interactive input available. Sandbox skipped.")
else:
    target_filename, target_name, exp_filename = None, None, None
    should_trim = True
    pdb_code = ""
    target_url = ""
    prediction_path = None
    prediction_only = False
    pred_label = "prediction"
    use_demo = False
    demo_message = ""

    if not user_input:
        use_demo = True
        demo_message = "[INFO] No choice entered, initiating demo run of 2OIU."
    else:
        if user_input.startswith("http://") or user_input.startswith("https://"):
            raw_input = '5'
            target_url = user_input
        else:
            raw_input = user_input.upper()

        if raw_input == '1':
            pdb_code = "1Y26"
            target_name = "Adenine Riboswitch"
            target_url = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/1y26_prediction.cif"
        elif raw_input == '2':
            pdb_code = "1EHZ"
            target_name = "Yeast tRNA-Phe"
            target_url = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/1ehz_prediction.cif"
        elif raw_input == '3':
            pdb_code = "4LVV"
            target_name = "THF Riboswitch"
            target_url = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/4lvv_prediction.cif"
        elif raw_input == '4':
            pdb_code = "2OEU"
            target_name = "Hammerhead Ribozyme"
            target_url = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/2oeu_prediction.cif"
        elif raw_input == '5':
            if not target_url:
                target_url = input("Enter the direct raw URL of your predicted RNA structure (mmCIF or PDB) [Press Enter for the 2OIU demo]: ").strip()

            if not target_url:
                use_demo = True
                demo_message = "[INFO] No URL entered, initiating demo run of 2OIU."
            else:
                prediction_path, url_problem = prepare_prediction(target_url)
                if url_problem:
                    use_demo = True
                    demo_message = (f"[INFO] Your prediction URL could not be used ({url_problem}), initiating demo run of 2OIU in place. "
                                    "Your prediction file is not used in this run.")
                else:
                    filename_base = os.path.basename(target_url).split('.')[0]
                    pred_label = re.sub(r"[^A-Za-z0-9_-]", "", filename_base)[:40] or "prediction"
                    suggested_pdb = filename_base.split('_')[0].upper()
                    if not re.fullmatch(r"[1-9][A-Z0-9]{3}", suggested_pdb):
                        suggested_pdb = ""

                    if suggested_pdb:
                        print(f"[TIP] Suggested comparison PDB based on filename: '{suggested_pdb}'")
                        pdb_code = input(f"Enter the 4-character experimental PDB ID [Press Enter to accept '{suggested_pdb}', or type none to view the prediction alone]: ").strip().upper()
                        if not pdb_code:
                            pdb_code = suggested_pdb
                    else:
                        pdb_code = input("Enter the 4-character PDB ID of the experimental structure [Press Enter to view the prediction alone]: ").strip().upper()
                    if pdb_code in ("NONE", "-"):
                        pdb_code = ""

                    if not pdb_code:
                        prediction_only = True
                        target_name = f"Prediction only: {pred_label}"
                        print("[INFO] No PDB ID entered: showing the prediction alone, without a comparison.")
                    elif not re.fullmatch(r"[1-9][A-Z0-9]{3}", pdb_code):
                        use_demo = True
                        demo_message = (f"[INFO] PDB file code not recognized ({pdb_code}), initiating demo run of 2OIU in place. "
                                        "Your prediction file is not used in this run.")
                    elif not reference_exists(pdb_code):
                        use_demo = True
                        demo_message = (f"[INFO] PDB file code not recognized (RCSB has no entry {pdb_code}), initiating demo run of 2OIU in place. "
                                        "Your prediction file is not used in this run.")
                    else:
                        target_name = f"Custom RNA Model vs {pdb_code}"
        else:
            use_demo = True
            demo_message = f"[INFO] Choice '{user_input}' not recognized, initiating demo run of 2OIU in place."

    if use_demo:
        print(demo_message)
        user_file_discarded = bool(target_url)
        pdb_code = DEMO_PDB
        target_url = DEMO_URL
        prediction_path = None
        prediction_only = False
        target_name = "2OIU demo (your prediction was not used)" if user_file_discarded else "2OIU demo"

    if prediction_only:
        os.makedirs("assets/report", exist_ok=True)
        os.makedirs("assets/figures", exist_ok=True)
        os.makedirs("assets/tables", exist_ok=True)

        target_filename = prediction_path
        print(f"\n[SANDBOX MODE: {target_name}]")

        p_struct = parse_structure_file(target_filename, f"pred_{pred_label}", keep_longest_chain_only=True)
        p_nt = count_valid_nucleotides(p_struct)

        print("--> Reading model confidence from the B-factor column...")
        df_metrics, mean_plddt = analyze_alphafold_model(target_filename)
        band_counts = {k: int(v) for k, v in df_metrics["Confidence_Band"].value_counts().items()} if not df_metrics.empty else {}

        pred_only_pdb = os.path.join("data", f"{pred_label}_prediction_only.pdb")
        pdb_writer = PDBIO()
        pdb_writer.set_structure(p_struct)
        pdb_writer.save(pred_only_pdb)

        print("--> Drawing the 2D diagram from the distance heuristic (experimental)...")
        pred_svg_b64, pred_db, pred_pairs, pred_seq = generate_2d_topology_svg(
            pred_only_pdb, f"assets/figures/coord_derived_pred_{pred_label}.svg")
        print("--> Computing the distance matrix (experimental)...")
        pred_heatmap_path, _ = compute_and_render_contact_map(pred_only_pdb, f"Pred_{pred_label}")
        pred_heatmap_b64 = image_to_base64(pred_heatmap_path)

        session_record = {
            "Mode": "Prediction only (no experimental structure compared)",
            "Prediction_Name": pred_label,
            "Prediction_Source": target_url,
            "Prediction_Nucleotides": p_nt,
            "Mean_pLDDT": mean_plddt,
            "Confidence_Band_Counts": band_counts,
            "Prediction_2D_BasePairs": pred_pairs
        }
        report_json_path = os.path.join("assets", f"prediction_only_{pred_label}.json")
        with open(report_json_path, "w") as f:
            json.dump(session_record, f, indent=4)
        print(f"[INFO] JSON summary saved to {report_json_path}")

        def viewer_text(path):
            """PDB text with registry residue names shown as A/C/G/U so the cartoon renders modified nucleotides."""
            out = []
            with open(path) as handle:
                for line in handle:
                    if line.startswith(("ATOM", "HETATM")):
                        canonical = RNA_HETATM_RES.get(line[17:20].strip())
                        line = "ATOM  " + line[6:17] + f"{canonical:>3}" + line[20:] if canonical else "ATOM  " + line[6:]
                    out.append(line)
            return "".join(out)

        confidence_text = "not available" if mean_plddt is None else f"{mean_plddt}"
        band_items = "".join(f"<li>{html_lib.escape(name)}: {count}</li>" for name, count in band_counts.items())

        ui_header_html = f"""
        <div style="background: #12121c; padding: 15px; border-radius: 8px; width: 920px; border: 1px solid #333; margin-bottom: 10px;">
            <h3 style="color: #00e5ff; margin-top: 0; margin-bottom: 10px; font-family: sans-serif; font-size: 16px;">3D view of the prediction (no experimental structure compared)</h3>
            <div style="background: #1e1e2f; color: #ffffff; padding: 10px; border-radius: 6px; text-align: center; border-left: 5px solid #0053D6; font-family: sans-serif;">
                <strong>Prediction: {pred_label}</strong><br>
                {PLDDT_LEGEND}
            </div>
        </div>
        """
        display(HTML(ui_header_html))

        try:
            import py3Dmol
            viewer = py3Dmol.view(width=920, height=480)
            viewer.addModel(viewer_text(pred_only_pdb), "pdb")
            style_by_plddt(viewer)
            viewer.zoomTo()
            viewer.show()

            dashboard_html = f"""
            <div style="width: 920px; font-family: sans-serif; margin-top: 20px; background: #12121c; padding: 20px; border-radius: 8px; border: 1px solid #333;">
                <h3 style="color: #00e5ff; border-bottom: 2px solid #333; padding-bottom: 8px; margin-top: 0;">2D drawing (distance heuristic, experimental) and distance matrix (experimental)</h3>
                <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #0053D6; margin-top: 0;">Predicted: heuristic pairs</h4>
                        <p style="color: #cbd5e1; font-size: 11px;">Pairs called by the heuristic (not a base-pair annotation): <strong>{pred_pairs}</strong></p>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px;">
                            <img src="data:image/svg+xml;base64,{pred_svg_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #0053D6; margin-top: 0;">Predicted distance matrix</h4>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px; display: inline-block;">
                            <img src="data:image/png;base64,{pred_heatmap_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                </div>
                <div style="font-family: monospace; font-size: 10px; background: #1e1e2f; color: #cbd5e1; padding: 10px; border-radius: 6px; word-break: break-all; border: 1px solid #444;">
                    <strong style="color: #00e5ff;">Sequence read by the drawing code:</strong> {pred_seq}
                </div>
            </div>
            """
            display(HTML(dashboard_html))

            metrics_summary_html = f"""
            <div style="width: 920px; background: #12121c; color: #ffffff; padding: 15px; border-radius: 8px; margin-top: 15px; font-family: sans-serif; border: 1px solid #333;">
                <h3 style="margin-top: 0; color: #00e5ff;">Prediction only: {pred_label}</h3>
                <ul style="list-style-type: none; padding: 0; line-height: 1.6; color: #cbd5e1;">
                    <li><strong>Source:</strong> {html_lib.escape(target_url)}</li>
                    <li><strong>Prediction nucleotides (longest RNA chain):</strong> {p_nt}</li>
                    <li><strong>Mean of the B-factor column:</strong> {confidence_text} (pLDDT only if the predictor stored it there)</li>
                    <li><strong>Nucleotides per band:</strong></li>
                </ul>
                <ul style="line-height: 1.4; color: #cbd5e1; margin-top: 0;">{band_items}</ul>
                <p style="color: #94a3b8; font-size: 12px;">No experimental structure was compared, so there is no RMSD and no sequence identity. To compare, run again and enter a PDB ID.</p>
            </div>
            """
            display(HTML(metrics_summary_html))
            display(HTML(COLOR_NOTE_HTML))

            html_report_path = os.path.join("assets", "report", f"viewer_snapshot_{pred_label}.html")
            full_report_content = f"""
            <html>
            <head><meta charset="utf-8"><title>FoldCheck-RNA report (prediction only) - {pred_label}</title></head>
            <body style="background: #0b0b10; padding: 20px;">
                <h2 style="color: #00e5ff; font-family: sans-serif; border-bottom: 2px solid #333; padding-bottom: 10px;">FoldCheck-RNA prediction report [{pred_label}]</h2>
                {ui_header_html}
                {viewer._make_html()}
                {dashboard_html}
                {metrics_summary_html}
                {COLOR_NOTE_HTML}
            </body>
            </html>
            """
            with open(html_report_path, "w", encoding="utf-8") as f_html:
                f_html.write(full_report_content)
            print(f"[INFO] HTML snapshot saved in {html_report_path}")

        except Exception as e:
            print(f"[WARNING] Visualizer or Dashboard error: {e}")

    elif pdb_code and len(pdb_code) == 4 and target_url:
        os.makedirs("data", exist_ok=True)
        os.makedirs("assets/report", exist_ok=True)
        os.makedirs("assets/figures", exist_ok=True)
        os.makedirs("assets/tables", exist_ok=True)

        exp_filename = os.path.join("data", f"EXP_{pdb_code}.cif")

        print(f"\n[SANDBOX MODE: {target_name}]")

        if prediction_path:
            target_filename = prediction_path
        else:
            target_filename = sanitize_and_validate_custom_url(target_url, os.path.join("data", f"target_{pdb_code}.cif"))
        download_file(f"https://files.rcsb.org/download/{pdb_code}.cif", exp_filename)

        p_struct = parse_structure_file(target_filename, f"pred_{pdb_code}", keep_longest_chain_only=True)
        e_struct = parse_structure_file(exp_filename, f"exp_{pdb_code}", keep_longest_chain_only=True)

        p_nt = count_valid_nucleotides(p_struct)
        e_nt = count_valid_nucleotides(e_struct)

        print(f"\n--> Analyzing RNA metrics for {target_name}...")
        df_metrics, mean_plddt = analyze_alphafold_model(target_filename)
        print("--> Running structural backbone alignment and superimposer...")

        rmsd_val, matched_res, seq_id, aligned_pred_file, aligned_ref_file, _, trim_rep = calculate_structural_rmsd_and_deviation(
            exp_filename, target_filename, trim_excess=should_trim
        )

        viewer_pred_pdb = aligned_pred_file
        viewer_ref_pdb = aligned_ref_file
        ref_chain_id = first_chain_id(viewer_ref_pdb)

        # 1. 2D drawings (distance heuristic) and sequences
        print("--> Drawing 2D diagrams from the distance heuristic (experimental) and extracting sequences...")
        pred_svg_path = f"assets/figures/coord_derived_pred_{pdb_code}.svg"
        exp_svg_path = f"assets/figures/coord_derived_exp_{pdb_code}.svg"

        pred_svg_b64, pred_db, pred_pairs, pred_seq = generate_2d_topology_svg(viewer_pred_pdb, pred_svg_path)
        exp_svg_b64, exp_db, exp_pairs, exp_seq = generate_2d_topology_svg(viewer_ref_pdb, exp_svg_path)

        seq_comparison_box = generate_highlighted_sequence_html(exp_seq, pred_seq)

        # 2. Distance matrices
        print("--> Computing distance matrices (experimental)...")
        pred_heatmap_path, _ = compute_and_render_contact_map(viewer_pred_pdb, f"Pred_{pdb_code}")
        exp_heatmap_path, _ = compute_and_render_contact_map(viewer_ref_pdb, f"Exp_{pdb_code}")

        pred_heatmap_b64 = image_to_base64(pred_heatmap_path)
        exp_heatmap_b64 = image_to_base64(exp_heatmap_path)

        session_record = {
            "Target_Name": target_name,
            "Reference_PDB": pdb_code,
            "Prediction_Nucleotides": p_nt,
            "Experimental_Nucleotides": e_nt,
            "Matched_Backbones": matched_res,
            "Sequence_Identity_Percent": round(seq_id, 1),
            "Global_RMSD_Angstroms": round(rmsd_val, 3),
            "Mean_pLDDT": mean_plddt,
            "Prediction_2D_BasePairs": pred_pairs,
            "Experimental_2D_BasePairs": exp_pairs
        }

        report_json_path = os.path.join("assets", f"validation_report_{pdb_code}.json")
        with open(report_json_path, "w") as f:
            json.dump(session_record, f, indent=4)
        print(f"[INFO] JSON summary saved to {report_json_path}")

        # --- Dark-Themed HTML UI Header for 3D Viewer ---
        ui_header_html = f"""
        <div style="background: #12121c; padding: 15px; border-radius: 8px; width: 920px; border: 1px solid #333; margin-bottom: 10px;">
            <h3 style="color: #00e5ff; margin-top: 0; margin-bottom: 10px; font-family: sans-serif; font-size: 16px;">3D superposition (prediction vs experimental structure)</h3>
            <div style="display: flex; justify-content: space-between; font-family: sans-serif;">
                <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 10px; border-radius: 6px; text-align: center; border-left: 5px solid #0053D6;">
                    <strong>Predicted structure (compared with {pdb_code})</strong><br>
                    {PLDDT_LEGEND}
                </div>
                <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 10px; border-radius: 6px; text-align: center; border-left: 5px solid #00e5ff;">
                    <strong>Experimental structure [{pdb_code}]</strong><br>
                    <small style="color: #cbd5e1;">Cartoon: <span style="color: #00e5ff;">&#9632;</span> chain {ref_chain_id} (cyan)</small>
                </div>
            </div>
        </div>
        """
        display(HTML(ui_header_html))

        try:
            import py3Dmol
            viewer = py3Dmol.view(viewergrid=(1, 2), width=920, height=480)

            def prep_pdb_for_viewer(filepath):
                mod_map = {
                    'PSU': 'U', 'H2U': 'U', '5MU': 'U', '4SU': 'U', 'OMC': 'C',
                    '5MC': 'C', '2MG': 'G', 'M2G': 'G', 'OMG': 'G', '7MG': 'G',
                    '1MA': 'A', 'YYG': 'G', 'INP': 'A', '5IU': 'U', '6MZ': 'A',
                    '2MA': 'A', '2M2': 'G', 'YG': 'G', '1MG': 'G', '5BU': 'U'
                }
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                patched = []
                for line in lines:
                    if line.startswith("ATOM") or line.startswith("HETATM"):
                        prefix = "ATOM  "
                        rest = line[6:]
                        resname = line[17:20].strip()
                        if resname in mod_map:
                            std_res = mod_map[resname]
                            line = prefix + rest[:11] + f"{std_res:>3}" + rest[14:]
                        else:
                            line = prefix + rest
                    patched.append(line)
                return "".join(patched)

            viewer.addModel(prep_pdb_for_viewer(viewer_pred_pdb), "pdb", viewer=(0, 0))
            style_by_plddt(viewer, (0, 0))

            viewer.addModel(prep_pdb_for_viewer(viewer_ref_pdb), 'pdb', viewer=(0, 1))
            viewer.setStyle({}, {'cartoon': {'color': '#00e5ff'}}, viewer=(0, 1))

            viewer.zoomTo()
            viewer.show()

            # --- Unified Dark-Themed Dashboard Section ---
            dashboard_html = f"""
            <div style="width: 920px; font-family: sans-serif; margin-top: 20px; background: #12121c; padding: 20px; border-radius: 8px; border: 1px solid #333;">

                <!-- Section 1: 2D Topologies -->
                <h3 style="color: #00e5ff; border-bottom: 2px solid #333; padding-bottom: 8px; margin-top: 0;">2D drawings from a distance heuristic (experimental)</h3>
                <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #0053D6; margin-top: 0;">Predicted: heuristic pairs</h4>
                        <p style="color: #cbd5e1; font-size: 11px;">Pairs called by the heuristic (not a base-pair annotation): <strong>{pred_pairs}</strong></p>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px;">
                            <img src="data:image/svg+xml;base64,{pred_svg_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #00e5ff; margin-top: 0;">Experimental: heuristic pairs</h4>
                        <p style="color: #cbd5e1; font-size: 11px;">Pairs called by the heuristic (not a base-pair annotation): <strong>{exp_pairs}</strong></p>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px;">
                            <img src="data:image/svg+xml;base64,{exp_svg_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                </div>

                <!-- Sequence Mismatch Highlighting Box -->
                <div style="background: #1e1e2f; padding: 15px; border-radius: 8px; border: 1px solid #444; margin-bottom: 25px;">
                    <h4 style="color: #00e5ff; margin-top: 0; margin-bottom: 5px;">Sequence alignment (mismatches highlighted)</h4>
                    <p style="color: #94a3b8; font-size: 11px; margin: 0;">Mismatches are highlighted; gaps are shown as dashes. This view recognises residues by name with a looser rule than the matching step, so it can include residues the RMSD does not use (for example a 5' GTP) and its identity can differ from the value above.</p>
                    {seq_comparison_box}
                </div>

                <!-- Section 2: 2D Distance Heatmaps -->
                <h3 style="color: #00e5ff; border-bottom: 2px solid #333; padding-bottom: 8px;">Distance matrices (experimental)</h3>
                <div style="display: flex; justify-content: space-between;">
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #0053D6; margin-top: 0;">Predicted Distance Matrix</h4>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px; display: inline-block;">
                            <img src="data:image/png;base64,{pred_heatmap_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                    <div style="width: 48%; background: #1e1e2f; color: #ffffff; padding: 15px; border-radius: 8px; text-align: center; border: 1px solid #444;">
                        <h4 style="color: #00e5ff; margin-top: 0;">Experimental Distance Matrix</h4>
                        <div style="background: #ffffff; padding: 10px; border-radius: 6px; margin-top: 10px; display: inline-block;">
                            <img src="data:image/png;base64,{exp_heatmap_b64}" style="max-width: 400px; height: auto;" />
                        </div>
                    </div>
                </div>
            </div>
            """
            display(HTML(dashboard_html))

            metrics_summary_html = f"""
            <div style="width: 920px; background: #12121c; color: #ffffff; padding: 15px; border-radius: 8px; margin-top: 15px; font-family: sans-serif; border: 1px solid #333;">
                <h3 style="margin-top: 0; color: #00e5ff;">Results ({pdb_code})</h3>
                <ul style="list-style-type: none; padding: 0; line-height: 1.6; color: #cbd5e1;">
                    <li><strong>Target:</strong> {target_name} vs Reference: {exp_filename}</li>
                    <li><strong>Prediction Nucleotides (Raw):</strong> {p_nt} | <strong>Experimental mmCIF Nucleotides:</strong> {e_nt}</li>
                    <li><strong>Matched nucleotide pairs:</strong> {matched_res} | <strong>Sequence identity (of matched pairs):</strong> {seq_id:.1f}%</li>
                    <li><strong>Global Backbone RMSD:</strong> {rmsd_val:.3f} Å</li>
                    <li><strong>Alignment notes:</strong> {trim_rep}</li>
                </ul>
            </div>
            """
            display(HTML(metrics_summary_html))
            display(HTML(COLOR_NOTE_HTML))

            html_report_path = os.path.join("assets", "report", f"viewer_snapshot_{pdb_code}.html")
            full_report_content = f"""
            <html>
            <head><meta charset="utf-8"><title>FoldCheck-RNA report - {pdb_code}</title></head>
            <body style="background: #0b0b10; padding: 20px;">
                <h2 style="color: #00e5ff; font-family: sans-serif; border-bottom: 2px solid #333; padding-bottom: 10px;">FoldCheck-RNA comparison report [{pdb_code}]</h2>
                {ui_header_html}
                {viewer._make_html()}
                {dashboard_html}
                {metrics_summary_html}
                {COLOR_NOTE_HTML}
            </body>
            </html>
            """

            with open(html_report_path, "w", encoding="utf-8") as f_html:
                f_html.write(full_report_content)
            print(f"[INFO] HTML snapshot saved in {html_report_path}")

        except Exception as e:
            print(f"[WARNING] Visualizer or Dashboard error: {e}")
    else:
        print("[WARNING] Invalid configuration or missing PDB code.")

print("\n" * 2)
print("[INFO] Sandbox finished.")
